In [1]:
import openai
from qdrant_client import QdrantClient

### Embedding Function

In [3]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


def get_embedding(text, model="gemini-embedding-001"):
    response = client.embeddings.create(
        model=model,
        input=text,
    )
    return response.data[0].embedding


vector = get_embedding("Random text")
print(len(vector))
print(vector[:5])

3072
[-0.014552347362041473, 0.03157435357570648, 0.012304088100790977, -0.04824463650584221, 0.0036319042555987835]


###  Retrieval Function

In [4]:
qdrant_client = QdrantClient(url='http://localhost:6333')

In [7]:
def retrieve_data(query, collection_name='amazon-items-collection-00', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }

In [8]:
retrieve_context = retrieve_data(query='Which case do you recommend for Iphone 12?')

In [9]:
retrieve_context


{'retrieved_context_ids': ['B0828S2KMV',
  'B0BYRWPV86',
  'B0BG6TMXDD',
  'B09XSBZQVS',
  'B0BMFYWK6T'],
 'retrieved_context_scores': [0.58709574,
  0.5791596,
  0.5741887,
  0.5590052,
  0.55822563],
 'retrieved_context_texts': ['Shechane AirPods 3 Case Cover 2021, Soft Leather Airpod Case with Strap Compatible with Airpod 3rd Generation, Supports Wireless Charging. (Black) 【Compatible with Airpods 3 case】: Specially designed for Airpod 3rd generation, not applicable for AirPods 1st, AirPods 2nd generation and AirPods Pro. 【Premium Material】: Made of high-quality genuine leather, Shechane AirPods 3 cases have unique natural texture and are soft, durable, delicate and stylish. 【Precise Design】: Precise cut-out ensures easy put-in and take-out of the charging case. And the accurate cut-out on the bottom supports USB charging as well as Wireless Charging. 【Full Protection】: Full-body covered soft leather and strong snap closure provide maximum protection for your Airpod 3 from dust, moi

### Format retrieved context function

In [10]:
def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context

In [11]:
formatted_context = process_context(retrieve_context)

In [12]:
print(formatted_context)

- Product ID: B0828S2KMV, Product Rating: 4.7, Product Description: Shechane AirPods 3 Case Cover 2021, Soft Leather Airpod Case with Strap Compatible with Airpod 3rd Generation, Supports Wireless Charging. (Black) 【Compatible with Airpods 3 case】: Specially designed for Airpod 3rd generation, not applicable for AirPods 1st, AirPods 2nd generation and AirPods Pro. 【Premium Material】: Made of high-quality genuine leather, Shechane AirPods 3 cases have unique natural texture and are soft, durable, delicate and stylish. 【Precise Design】: Precise cut-out ensures easy put-in and take-out of the charging case. And the accurate cut-out on the bottom supports USB charging as well as Wireless Charging. 【Full Protection】: Full-body covered soft leather and strong snap closure provide maximum protection for your Airpod 3 from dust, moisture, scratches ,drops and shocks. 【Easy to carry】: Strongly sewn with a leather strap, Shechane airpod 3 case can be hung on a handbag, backpack, or a key chain, 

### Build the prompt template

In [15]:
def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt

In [16]:
prompt = build_prompt(question='Do you have a case for Iphone 12?', formatted_context=formatted_context)

In [17]:
print(prompt)


    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    - Product ID: B0828S2KMV, Product Rating: 4.7, Product Description: Shechane AirPods 3 Case Cover 2021, Soft Leather Airpod Case with Strap Compatible with Airpod 3rd Generation, Supports Wireless Charging. (Black) 【Compatible with Airpods 3 case】: Specially designed for Airpod 3rd generation, not applicable for AirPods 1st, AirPods 2nd generation and AirPods Pro. 【Premium Material】: Made of high-quality genuine leather, Shechane AirPods 3 cases have unique natural texture and are soft, durable, delicate and stylish. 【Precise Design】: Precise cut-out ensures easy put-in and take-out of the charging case. And the accurate cut-out on the bottom supp

###  Generate the answer function

In [18]:
import os
from groq import Groq

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))


def generate_answer(prompt):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        reasoning_effort="low",
        max_completion_tokens=1000,
    )

    return response.choices[0].message.content


print(generate_answer("Say hello in one short sentence."))

Hello!


In [19]:
print(generate_answer(prompt))

We don’t have a case for the iPhone 12 among the available products.


###  Compile RAG pipeline

In [20]:
def rag_pipeline(question, topk=5):
    retrieve_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieve_context)

    prompt = build_prompt(question=question, formatted_context=formatted_context)
    answer = generate_answer(prompt)

    return answer


In [22]:
print(rag_pipeline('What kind of earphones can i get  with rating above 4.5', 10))

You can choose from the following earphones that have a rating higher than 4.5:

- **Product ID B0CH8DRD6K** – 2‑Pack iPhone wired headphones with a 3.5 mm jack, Apple MFi certification, built‑in microphone and remote control, ergonomic semi‑in‑ear design and compatibility with iPhone, iPad, iPod, Android, MP3/4 devices. Rating 4.9.

- **Product ID B0BM657X74** – Bluet00th sleep headphones headband, wireless, ultra‑thin earbuds built into a comfortable stretchable headband, Hi‑Fi stereo sound, built‑in microphone and volume controls, 10–12 hours of playtime, washable fabric. Rating 4.6.

- **Product ID B0BMFYWK6T** – 4‑pack silicone covers for the 3rd‑gen Fire TV remote (covers only, not earphones). Rating 4.6.
